In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
df_order = spark.read.table("olist.bronze.olist_order")

In [0]:
df_order = spark.read.table("olist.bronze.olist_order")
# remove the double quotes from column names
def clean_column(df):
    new_columns = [
        col.replace('"', '') for col in df.columns
    ]
    return df.toDF(*new_columns)
df_order = clean_column(df_order)



In [0]:
# Convert string columns to timestamp format
timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in timestamp_columns:
    df_order = df_order.withColumn(
        col, 
        F.when(F.col(col) == "", None).otherwise(F.to_timestamp(F.col(col)))
    )

In [0]:
# Fill nulls only in string columns (not timestamp columns)
string_columns = ["order_id", "customer_id", "order_status"]
for col in string_columns:
    df_order = df_order.withColumn(col, F.coalesce(F.col(col), F.lit("unknown")))

In [0]:
df_order.display()

In [0]:
df_order.write.format("delta").mode("overwrite").saveAsTable("olist.silver.olist_order")